# Day 5 - Neural Networks and ANN (Keras)

Warm up on MNIST, then train a small feed-forward churn model with normalized inputs and early stopping. Compare it with the Phase 1 logistic champion on the same untouched holdout. This notebook prepares its own data; it does not require running earlier notebooks.

In [3]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import tensorflow as tf

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.ann import build_churn_ann
from src.feature_engineering import ChurnInputCleaner

## Keras warm-up: MNIST

A compact dense network demonstrates image normalization, dense layers, softmax classification, and a basic training loop.

In [4]:
(x_mnist_train, y_mnist_train), (x_mnist_test, y_mnist_test) = tf.keras.datasets.mnist.load_data()
x_mnist_train = x_mnist_train.astype("float32") / 255.0
x_mnist_test = x_mnist_test.astype("float32") / 255.0

mnist_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax"),
])
mnist_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
mnist_model.fit(x_mnist_train, y_mnist_train, epochs=3, batch_size=128, validation_split=0.1)
print("MNIST test metrics:", mnist_model.evaluate(x_mnist_test, y_mnist_test, verbose=0))

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
Epoch 1/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8946 - loss: 0.3800 - val_accuracy: 0.9565 - val_loss: 0.1709
Epoch 2/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9504 - loss: 0.1730 - val_accuracy: 0.9667 - val_loss: 0.1231
Epoch 3/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9650 - loss: 0.1242 - val_accuracy: 0.9708 - val_loss: 0.1032
MNIST test metrics: [0.11164438724517822, 0.9672999978065491]


## Telco churn dataset

Use the same stratified 80:20 split (random state 42) as the earlier increments. `TotalCharges` is cleaned before the split; the saved input pipeline also handles raw API input.

In [5]:
from sklearn.model_selection import train_test_split

raw = pd.read_csv(PROJECT_ROOT / "data" / "churn.csv")
raw["TotalCharges"] = pd.to_numeric(raw["TotalCharges"].astype(str).str.strip(), errors="coerce").fillna(0.0)
raw = raw.drop(columns=["customerID"])
raw["Churn"] = raw["Churn"].map({"No": 0, "Yes": 1}).astype("int8")
X = raw.drop(columns=["Churn"])
y = raw["Churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train/test rows:", len(X_train), len(X_test))
print("Churn prevalence:", round(float(y.mean()), 4))

Train/test rows: 5634 1409
Churn prevalence: 0.2654


## Preprocess and train the ANN

Scale numeric columns and one-hot encode categorical columns. Early stopping monitors validation loss and restores the best weights; the fixed test set stays out of training and model selection.

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# Keep an explicit validation partition inside the training data for early stopping.
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42, stratify=y_train
)
numeric_columns = X_fit.select_dtypes(include="number").columns.tolist()
categorical_columns = X_fit.select_dtypes(exclude="number").columns.tolist()
ann_preprocessor = Pipeline([
    ("input_cleaner", ChurnInputCleaner()),
    ("columns", ColumnTransformer([
        ("numeric", StandardScaler(), numeric_columns),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_columns),
    ], remainder="drop")),
])

x_fit = ann_preprocessor.fit_transform(X_fit).astype("float32")
x_val = ann_preprocessor.transform(X_val).astype("float32")
x_test = ann_preprocessor.transform(X_test).astype("float32")
y_fit_array = y_fit.to_numpy(dtype="float32")
y_val_array = y_val.to_numpy(dtype="float32")
y_test_array = y_test.to_numpy(dtype="int8")

classes = np.unique(y_fit_array).astype("int64")
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_fit_array.astype("int64"))
class_weights = {int(label): float(weight) for label, weight in zip(classes, weights)}

tf.keras.utils.set_random_seed(42)
ann_model = build_churn_ann(input_dim=x_fit.shape[1], hidden_units=(64, 32), dropout_rate=0.2)
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True
)
history = ann_model.fit(
    x_fit, y_fit_array,
    validation_data=(x_val, y_val_array),
    epochs=100,
    batch_size=32,
    class_weight=class_weights,
    callbacks=[early_stopping],
    verbose=1,
)
best_epoch = int(np.argmin(history.history["val_loss"]) + 1)
print("Early stopping selected epoch:", best_epoch)

# Refit preprocessing and the ANN on all training rows at the selected epoch count.
# This gives the ANN the same training partition as the classical champion.
ann_preprocessor.fit(X_train, y_train)
x_train_full = ann_preprocessor.transform(X_train).astype("float32")
x_test = ann_preprocessor.transform(X_test).astype("float32")
y_train_array = y_train.to_numpy(dtype="float32")
classes = np.unique(y_train_array).astype("int64")
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train_array.astype("int64"))
class_weights = {int(label): float(weight) for label, weight in zip(classes, weights)}

tf.keras.utils.set_random_seed(42)
ann_model = build_churn_ann(input_dim=x_train_full.shape[1], hidden_units=(64, 32), dropout_rate=0.2)
ann_model.fit(
    x_train_full, y_train_array,
    epochs=best_epoch,
    batch_size=32,
    class_weight=class_weights,
    verbose=1,
)

Epoch 1/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.7066 - auc: 0.7819 - loss: 0.5619 - precision: 0.4664 - recall: 0.7317 - val_accuracy: 0.7577 - val_auc: 0.8399 - val_loss: 0.4833 - val_precision: 0.5292 - val_recall: 0.7679
Epoch 2/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7425 - auc: 0.8295 - loss: 0.5077 - precision: 0.5097 - recall: 0.7876 - val_accuracy: 0.7624 - val_auc: 0.8420 - val_loss: 0.4827 - val_precision: 0.5354 - val_recall: 0.7768
Epoch 3/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7492 - auc: 0.8361 - loss: 0.4983 - precision: 0.5178 - recall: 0.7994 - val_accuracy: 0.7636 - val_auc: 0.8426 - val_loss: 0.4717 - val_precision: 0.5377 - val_recall: 0.7634
Epoch 4/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7546 - auc: 0.8419 - loss: 0.4906 - precision: 0.5249 - recall: 0.7954 - val_accuracy: 0.7600 - val_auc: 0.8434 - val_loss: 0.4715 - val_precision: 0.5325 - val_recall: 0.7679
Epoch 5/100
150/150 ━━━━━━━━

## Compare with the Phase 1 champion

The logistic champion is rebuilt with its recorded settings on the same training partition so this notebook stays independent. The original test partition is evaluated once after ANN training.

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.pipeline import Pipeline

# Day 4 retained the Day 3 champion: balanced logistic regression (C=1).
champion_preprocessor = Pipeline([
    ("input_cleaner", ChurnInputCleaner()),
    ("columns", ColumnTransformer([
        ("numeric", StandardScaler(), numeric_columns),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
    ], remainder="drop")),
])
champion = Pipeline([
    ("preprocessor", champion_preprocessor),
    ("classifier", LogisticRegression(C=1, class_weight="balanced", max_iter=2000, random_state=42)),
])
champion.fit(X_train, y_train)
champion_probability = champion.predict_proba(X_test)[:, 1]
ann_probability = ann_model.predict(x_test, verbose=0).reshape(-1)


def score_model(name, truth, probability):
    predicted = (probability >= 0.5).astype("int8")
    return {
        "Model": name,
        "Accuracy": accuracy_score(truth, predicted),
        "Precision": precision_score(truth, predicted, zero_division=0),
        "Recall": recall_score(truth, predicted, zero_division=0),
        "ROC-AUC": roc_auc_score(truth, probability),
    }

benchmark = pd.DataFrame([
    score_model("Phase 1 champion - Logistic Regression", y_test_array, champion_probability),
    score_model("Day 5 ANN", y_test_array, ann_probability),
]).set_index("Model")
display(benchmark.round(4))

recall_delta = benchmark.loc["Day 5 ANN", "Recall"] - benchmark.loc["Phase 1 champion - Logistic Regression", "Recall"]
auc_delta = benchmark.loc["Day 5 ANN", "ROC-AUC"] - benchmark.loc["Phase 1 champion - Logistic Regression", "ROC-AUC"]
print(f"ANN minus champion: recall {recall_delta:+.4f}; ROC-AUC {auc_delta:+.4f}")
if recall_delta > 0 and auc_delta > 0:
    print("On this fixed holdout, the ANN improved both churn recall and ROC-AUC. Review precision and validation stability before promoting it.")
elif recall_delta > 0:
    print("On this fixed holdout, the ANN improved primary churn recall but reduced ROC-AUC; review the tradeoff before promoting it.")
elif auc_delta > 0:
    print("On this fixed holdout, the ANN did not improve primary churn recall, though ROC-AUC increased; depth improved ranking quality without improving the selected threshold metric.")
else:
    print("On this fixed holdout, the ANN did not improve either churn recall or ROC-AUC; added depth has not shown value on this comparison.")

,Accuracy,Precision,Recall,ROC-AUC
Model,,,,
Phase 1 champion - Logistic Regression,0.7381,0.5043,0.7834,0.8416
Day 5 ANN,0.7544,0.5268,0.7353,0.8383


ANN minus champion: recall -0.0481; ROC-AUC -0.0033
On this fixed holdout, the ANN did not improve either churn recall or ROC-AUC; added depth has not shown value on this comparison.


## Save and use the ANN

Save the network and its input transformer separately so inference uses exactly the same feature scaling and encoding. The service keeps the classical champion as its default; call `predict(..., model_name="ann")` to opt into this challenger.

In [8]:
import joblib

model_dir = PROJECT_ROOT / "models"
model_dir.mkdir(parents=True, exist_ok=True)
ann_model_path = model_dir / "churn_ann_day5.keras"
ann_preprocessor_path = model_dir / "churn_ann_preprocessor.joblib"
ann_model.save(ann_model_path)
joblib.dump(ann_preprocessor, ann_preprocessor_path)
print("Saved ANN:", ann_model_path)
print("Saved preprocessing pipeline:", ann_preprocessor_path)

Saved ANN: C:\Users\Abhishek\Desktop\RetailMind-AI\models\churn_ann_day5.keras
Saved preprocessing pipeline: C:\Users\Abhishek\Desktop\RetailMind-AI\models\churn_ann_preprocessor.joblib


In [9]:
from src.predict import predict

# Exercise single-customer and batch inference through the reusable service.
display(predict(X_test.head(), model_name="ann"))
display(predict(X_test.iloc[0].to_dict(), model_name="ann"))

,churn_prediction,churn_probability
437,No,0.071059
2280,Yes,0.867926
2235,No,0.124928
4460,No,0.421090
3761,No,0.044691


,churn_prediction,churn_probability
0,No,0.071059
